In [1]:
!pip install genai-processors

In [2]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('')

In [3]:
import asyncio
from genai_processors import content_api
from genai_processors import processor
from genai_processors.core import pdf

In [16]:
from google import genai
import google.generativeai as legacy_genai

In [5]:
import csv, time, os
import hashlib, statistics as stats

from datetime import datetime, timezone

from typing import List, Dict, Any

ReportLab
https://github.com/google-gemini/genai-processors/blob/main/notebooks/create_your_own_processor.ipynb

In [6]:
output_csv_path = 'PDF_extraction_results.csv'

class PDFProcessor:
  def __init__(self, pdf_path: str, output_csv=output_csv_path):
    self.pdf_path = pdf_path
    self.output_csv = output_csv
    self.extracted_data:List[Dict[str, Any]] = []
    self.extracted_content:List[Dict[str, Any]] = []

  async def extract_content(self, pdf_bytes: bytes) -> None:
    extractor = pdf.PDFExtract()
    metadata = {'original_file_name': os.path.basename(self.pdf_path)}

    async for part in extractor(
        content_api.ProcessorPart(
            pdf_bytes,
            mimetype=pdf.PDF_MIMETYPE,
            metadata=metadata
        )
    ):
      self.extracted_content.append(part)
      # print("type(part):", type(part))
      print('metadata:', getattr(part, '__dict__', 'no_dict'))
      print("text:", getattr(getattr(part, "_part", "NO_part"), 'text',''))

      text = getattr(getattr(part, '_part', 'no_part'), 'text','')
      print(f'{time.perf_counter()} - Processing part: {part}')

      raw_content = getattr(part, '_part', b"")
      if isinstance(raw_content, bytes):
        preview = str(text)[:200]
        # preview = raw_content[:200].decode('utf-8', errors='ignore')
        out_len = len(raw_content)
      else:
        preview = str(text)[:200]
        out_len = len(str(raw_content))

      self.extracted_data.append({
          'id': part.metadata.get('id', ''),
          'original_file_name': part.metadata.get('original_file_name', ''),
          'cache_hit': part.metadata.get('cache_hit', ''),
          'page_number': part.metadata.get('page_number', ''),
          'preview': preview,
          'out_len': out_len,
      })

  def write_results_to_csv(self) -> None:
      if not self.extracted_data:
        print("No data extracted to write to CSV.")
        return

      try:
        fieldnames = list(self.extracted_data[0].keys())
        file_exists = os.path.exists(self.output_csv)

        with open(self.output_csv, 'a', newline='', encoding='utf-8') as f:
          w = csv.DictWriter(f, fieldnames=fieldnames)
          if not file_exists:
            w.writeheader()
          w.writerows(self.extracted_data)
        print(f'{time.perf_counter():.3f} - done csv: {self.output_csv}')

      except Exception as e:
        print(f"An error occurred during CSV write: {e}")

  def data_present(self):
      import pandas as pd
      return pd.DataFrame([p.__dict__ for p in self.extracted_content] if self.extracted_content else [])

  async def run(self) -> None:
    print(f'pdf processing on {self.pdf_path}')

    if not os.path.exists(self.pdf_path):
      print(f'e: pdf_path not found: {self.pdf_path}')
      return

    try:
      with open(self.pdf_path, 'rb') as f:
        pdf_bytes = f.read()
      await self.extract_content(pdf_bytes)
    except Exception as e:
      print(f'e: {e}')
    finally:
      self.write_results_to_csv()

In [7]:
pdf_file_path = '/content/'
operator = PDFProcessor(pdf_file_path)
await operator.run()

pdf processing on /content/NextjsProseMirror.pdf
metadata: {'_metadata': {}, '_part': Part(
  text='Parsed PDF NextjsProseMirror.pdf (17 pages, 17 pages with images)'
), '_role': '', '_substream_name': 'status', '_mimetype': 'text/plain'}
text: Parsed PDF NextjsProseMirror.pdf (17 pages, 17 pages with images)
2020.450624878 - Processing part: ProcessorPart({'text': 'Parsed PDF NextjsProseMirror.pdf (17 pages, 17 pages with images)'}, mimetype='text/plain', substream_name='status')
metadata: {'_metadata': {}, '_part': Part(
  text="""--- START OF PDF NextjsProseMirror.pdf ---

"""
), '_role': '', '_substream_name': '', '_mimetype': 'text/plain'}
text: --- START OF PDF NextjsProseMirror.pdf ---


2020.450800473 - Processing part: ProcessorPart({'text': '--- START OF PDF NextjsProseMirror.pdf ---\n\n'}, mimetype='text/plain')
metadata: {'_metadata': {}, '_part': Part(
  text="""---- Screenshot for PAGE 1 ----

"""
), '_role': '', '_substream_name': '', '_mimetype': 'text/plain'}
text: ---

In [8]:
df_results = operator.data_present()
display(df_results)

,_metadata,_part,_role,_substream_name,_mimetype
0,{},media_resolution=None code_execution_result=No...,,status,text/plain
1,{},media_resolution=None code_execution_result=No...,,,text/plain
2,{},media_resolution=None code_execution_result=No...,,,text/plain
3,{},media_resolution=None code_execution_result=No...,,,image/webp
4,{},media_resolution=None code_execution_result=No...,,,text/plain
...,...,...,...,...,...
65,{},media_resolution=None code_execution_result=No...,,,text/plain
66,{},media_resolution=None code_execution_result=No...,,,text/plain
67,{},media_resolution=None code_execution_result=No...,,,image/webp
68,{},media_resolution=None code_execution_result=No...,,,text/plain


In [9]:
import pandas as pd
if os.path.exists(output_csv_path):
    csv_df = pd.read_csv(output_csv_path)
    display(csv_df.tail())
    csv_preview = "\n".join(csv_df['preview'].astype(str).tolist())
    csv_preview = csv_preview[:1000]
    print(len(csv_preview))

else:
    print('CSV file not found.')

,id,original_file_name,cache_hit,page_number,preview,out_len
65,NaN,NaN,NaN,NaN,"이 구조가 좋아지는 순간은 “툴바 active 상태”, “이미지 업로드 placeh...",1337
66,NaN,NaN,NaN,NaN,---- Screenshot for PAGE 17 ----\n\n,243
67,NaN,NaN,NaN,NaN,NaN,516
68,NaN,NaN,NaN,NaN,--- PAGE 17 ----\n\n,227
69,NaN,NaN,NaN,NaN,"6. React UI\r\n다음 단계로는 툴바 active 상태 계산, 멘션 노드 ...",321


1000


In [34]:
class CustomPDFOperator():

  def __init__(self, bench_csv='bench.csv'):
    super().__init__()
    self.bench_csv = bench_csv
    self.l1_cache:dict[str, str] = {}

  def pctl(self, xs:list[float], q:int) -> float:
    if not xs:
      return 0.0
    xs = sorted(xs); k = int(round((len(xs) - 1) * q / 100))
    return xs[k]

  def ask(self, q:str) -> tuple[str, float, bool, tuple[int, int, int]]:
    api_key = userdata.get('')
    legacy_genai.configure(api_key=api_key)
    model = legacy_genai.GenerativeModel('')

    key = hashlib.sha256(q.encode('utf-8')).hexdigest()

    if key in self.l1_cache:
      cache_text = self.l1_cache[key]
      return cache_text, 0.0, True, (0, 0, 0)

    t0 = time.perf_counter()
    r = model.generate_content(q)
    dt = (time.perf_counter() - t0) * 1000

    u = getattr(r, 'usage_metadata', None)
    usage = (
        getattr(u, 'prompt_token_count', 0),
        getattr(u, 'candidates_token_count', 0),
        getattr(u, 'total_token_count', 0)
    )
    text = r.text
    print(text)
    self.l1_cache[key] = text

    return text, dt, False, usage

  def run(self):
    prompt = f"The following is content extracted from the CSV. Please summarize it:\n\n{csv_preview}"

    response_text, duration, cached, usage = self.ask(prompt)
    print('Response:', response_text[:100], '...')

    lats, hits = [], 0

    current_items = list(self.l1_cache.items())
    for i, (key, val) in enumerate(current_items, 1):
      text, dt, hit, use = self.ask(val)
      lats.append(dt)
      if hit: hits += 1

    if lats:
      p95 = self.pctl([x for x in lats if x > 0], 95)
      print(f'Done. p95 Latency: {p95:.2f} ms, Cache Hits: {hits}')

    return response_text[:100]

In [35]:
pdf_operator = CustomPDFOperator()
pdf_operator.run()

The PDF "NextjsProseMirror.pdf" (17 pages, in Korean) outlines the process of integrating ProseMirror into Next.js applications.

It emphasizes three core principles:
1.  **Client-only initialization**
2.  **Effective SSR (Server-Side Rendering) separation**
3.  **Proper management of the EditorView lifecycle**

The content includes practical examples, such as implementing ProseMirror within a Next.js client component (`app/editor/page.tsx`) using `useEffect` and `useRef`. To achieve complete SSR separation, the guide advises using `next/dynamic` with `ssr: false` for client-side-only rendering of the editor. It also briefly discusses the safety of this architectural approach and refers to basic ProseMirror examples.
Response: The PDF "NextjsProseMirror.pdf" (17 pages, in Korean) outlines the process of integrating ProseMirro ...
The 17-page Korean PDF "NextjsProseMirror.pdf" provides a comprehensive guide for integrating the ProseMirror rich text editor into Next.js applications, with

'The PDF "NextjsProseMirror.pdf" (17 pages, in Korean) outlines the process of integrating ProseMirro'

In [ ]:
def ask_with_pdf(pdf_path, user_query):
  sample_file = genai.upload_file(path=pdf_path, display_name="Summary Target")

  r = model.generate_content([sample_file, user_query])

  return r.text

In [36]:
from google import genai
import os

# Initialize the client using the API key
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Hello, how are you?"
)

print(response.text)

Hello! As an AI, I don't have feelings in the way humans do, but I'm ready and functioning well.

How are you doing today?


In [37]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from datetime import datetime

from uuid import uuid4

app = FastAPI(title='Gemini Genai Session')

sessions: dict[str, list[dict]] = {}

In [22]:
class QueryRequest(BaseModel):
  query: str
  session_id: str | None = None

class QueryResponse(BaseModel):
  session_id: str
  query: str
  answer: str
  timestamp: str
  turn_count: int

class QueryTurn(BaseModel):
  query: str
  answer: str
  timestamp: str

In [23]:
def simple_answer(query: str) -> str:
  q = query.strip().lower()

  if not q:
    return 'Empty query received'
  if 'pdf' in q:
    try:
      result = pdf_operator.run()
      return f'PDF processing complete. {result}'
    except Exception as e:
      return f'error: {e}'

  return f'answer for {query}'

In [24]:
@app.get("/")
async def root():
    return {"message": "API running", "docs": "/docs"}

In [25]:
@app.post('/query', response_model=QueryResponse)
async def query_api(req: QueryRequest):
  session_id = req.session_id or str(uuid4())
  answer = simple_answer(req.query)
  timestamp = datetime.utcnow().isoformat()

  if session_id not in sessions:
    sessions[session_id] = []

  sessions[session_id].append({
      'query': req.query,
      'answer': answer,
      'timestamp': timestamp,
  })

  return QueryResponse(
      session_id= session_id,
      query= req.query,
      answer= answer,
      timestamp=timestamp,
      turn_count= len(sessions[session_id]),
  )

In [38]:
@app.get('/sessions/{session_id}')
async def get_session(session_id: str):
  if session_id not in sessions:
    raise HTTPException(status_code=404, detail='Session not found')

  return {
      'session_id': session_id,
      'history': sessions.get(session_id, [])
  }

In [39]:
import httpx

async def test_get_session():

    test_id = "test-uuid-123"
    req = QueryRequest(query="parsing_pdf")
    res = simple_answer(req.query)
    sessions[test_id] = [res]

    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://testserver") as client:
        response = await client.get(f"/sessions/{test_id}")

    print(f"Status Code: {response.status_code}")
    display(response.json())

# Run the test
await test_get_session()

Response: The PDF "NextjsProseMirror.pdf" (17 pages, in Korean) outlines the process of integrating ProseMirro ...
The 17-page Korean PDF "NextjsProseMirror.pdf" outlines a comprehensive strategy for integrating the ProseMirror rich text editor into Next.js applications, particularly focusing on optimal performance and compatibility within an SSR (Server-Side Rendering) environment using the App Router.

The document is structured around three critical principles:

1.  **Client-only Initialization:** Acknowledging ProseMirror's direct DOM manipulation requirements, the guide leverages React's `useEffect` hook to ensure the editor's setup code executes exclusively on the client side after the initial render. `useRef` is employed to maintain stable references to both the editor's DOM container and the `EditorView` instance.
2.  **Effective SSR Separation:** To prevent server-side execution errors for client-specific ProseMirror code, the PDF strongly advocates using `next/dynamic` with t

{'session_id': 'test-uuid-123',
 'history': ['PDF processing complete. The PDF "NextjsProseMirror.pdf" (17 pages, in Korean) outlines the process of integrating ProseMirro']}

In [ ]:
import json

output_file = 'sessions_export.json'

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(sessions, f, ensure_ascii=False, indent=4)

print(f'Sessions successfully saved to {output_file}')

Sessions successfully saved to sessions_export.csv
